# 05 — Gold dimensions (date and station) · df

Builds the two small conformed dimensions the Direct Lake semantic model needs.
They are **physical Delta tables in `lh_gold`** rather than DAX calculated tables,
because Direct Lake calculated tables/columns are still preview and can cause
unexpected fallback.

**Outputs**

| Table | Grain | Purpose |
|---|---|---|
| `gold_df_dim_date` | one row per calendar date in the gold window | report calendar, month/week grouping, weekday |
| `gold_df_dim_station` | one row per flow station | shared station labels, asset and owner |

**Rules**

1. The date range is taken from `gold_risk_daily`, so the calendar always matches
   the gold grid — no hard-coded years.
2. The station list is taken from `gold_risk_daily` (the complete grid), not from
   observed rows, so every station survives a day with no feed.
3. Both tables are rebuilt deterministically on every run. They carry no business
   facts, so overwriting is safe.


In [ ]:
from datetime import datetime, timezone
from pyspark.sql import functions as F

# --- sandbox identifiers ---------------------------------------------------
WORKSPACE_ID = "355cd573-8b17-430a-b606-c6b9424d36a1"
GOLD_ID = "df9a08e5-b370-4c2a-b608-6d0f326003ef"

GOLD = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{GOLD_ID}"

BATCH_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
COMPUTED_AT = datetime.now(timezone.utc).replace(tzinfo=None)


def read_gold(name: str):
    return spark.read.format("delta").load(f"{GOLD}/Tables/{name}")


def write_gold(name: str, sdf) -> None:
    path = f"{GOLD}/Tables/{name}"
    (sdf.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(path))
    spark.sql(f"CREATE TABLE IF NOT EXISTS {name} USING DELTA LOCATION '{path}'")
    print(f"{name}: {sdf.count():,} rows")


print(f"Batch: {BATCH_ID}")


In [ ]:
# --- 1. calendar dimension -------------------------------------------------
# The pre-df dimension tables were a naming rehearsal. Drop them so the
# serving layer has one canonical set of dimensions.
for _old in ("gold_dim_date", "gold_dim_station"):
    spark.sql(f"DROP TABLE IF EXISTS {_old}")

import pandas as pd

risk = read_gold("gold_risk_daily").select(
    "date", "station_id", "station_name", "asset", "owner_name", "owner_upn"
)

bounds = risk.agg(
    F.min("date").alias("min_date"),
    F.max("date").alias("max_date"),
).first()
MIN_DATE, MAX_DATE = bounds["min_date"], bounds["max_date"]

# Build the date spine in pandas so it does not depend on Spark sequence/interval
# support in the runtime. 90-ish rows, so this is trivial.
spine = pd.DataFrame({
    "date": pd.date_range(str(MIN_DATE), str(MAX_DATE), freq="D")
})
dim_date = (
    spark.createDataFrame(spine)
    .select(
        F.col("date").cast("date").alias("date"),
    )
    .select(
        "date",
        F.year("date").cast("int").alias("year"),
        F.quarter("date").cast("int").alias("quarter"),
        F.month("date").cast("int").alias("month"),
        F.date_format("date", "MMM").alias("month_name"),
        F.date_format("date", "yyyy-MM").alias("year_month"),
        F.date_sub("date", ((F.dayofweek("date") + 5) % 7)).alias("week_start"),
        F.weekofyear("date").cast("int").alias("week_of_year"),
        F.date_format("date", "EEEE").alias("weekday"),
        ((F.dayofweek("date") + 5) % 7).cast("int").alias("weekday_num"),
        ((F.dayofweek("date") + 5) % 7 >= 5).alias("is_weekend"),
        F.dayofmonth("date").cast("int").alias("day_of_month"),
        F.dayofyear("date").cast("int").alias("day_of_year"),
    )
    .orderBy("date")
)

print(f"date window: {MIN_DATE} -> {MAX_DATE}")
write_gold("gold_df_dim_date", dim_date)
display(dim_date.limit(7).toPandas())


In [ ]:
# --- 2. station dimension --------------------------------------------------
dim_station = (
    risk.select("station_id", "station_name", "asset", "owner_name", "owner_upn")
    .dropDuplicates(["station_id"])
    .orderBy("station_id")
)

# Fail loudly if a station appears twice or an id is missing.
n_stations = dim_station.count()
n_ids = dim_station.select("station_id").distinct().count()
assert n_stations == n_ids, "duplicate station_id in gold_df_dim_station"
assert dim_station.filter(F.col("station_id").isNull()).count() == 0, "null station_id"

write_gold("gold_df_dim_station", dim_station)
display(dim_station.toPandas())


In [ ]:
# --- 3. verification -------------------------------------------------------
expected_days = (MAX_DATE - MIN_DATE).days + 1
n_days = read_gold("gold_df_dim_date").count()
n_unique_days = read_gold("gold_df_dim_date").select("date").distinct().count()
n_station_rows = read_gold("gold_df_dim_station").count()
risk_stations = risk.select("station_id").distinct().count()

assert n_days == expected_days, f"calendar rows {n_days} != expected {expected_days}"
assert n_unique_days == n_days, "duplicate dates in gold_df_dim_date"
assert n_station_rows == risk_stations, (
    f"station rows {n_station_rows} != risk stations {risk_stations}"
)

print(f"calendar rows      : {n_days:,} ({MIN_DATE} -> {MAX_DATE})")
print(f"calendar unique    : {n_unique_days:,}")
print(f"station rows       : {n_station_rows:,}")
print(f"risk stations      : {risk_stations:,}")
print("dimension checks   : OK")
